# 第五章关键结论的数值复核

G8｜C1组

受审版本：`bd5f7307670d618e0caca967f103126e2ad52c8c`。
本实验检查固定版本中的二分误差界、迭代稳定性、曲率条件和非线性最小二乘步长。运行时间与环境以末尾的运行记录为准；运行本组基准不等于重新执行全部 A/B 材料。工具使用情况见 AI 对话记录。

In [1]:
import ast,json,math,sys,platform
from pathlib import Path
import numpy as np
import scipy
from scipy.optimize import brentq
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
OUT=Path('图片');OUT.mkdir(exist_ok=True)
results={}
def rec(k,v):
    results[k]=v;print(k,json.dumps(v,ensure_ascii=False,indent=2))
print('Python',sys.version.split()[0],'NumPy',np.__version__,'SciPy',scipy.__version__)

Python 3.12.14 NumPy 1.26.4 SciPy 1.14.1


## 5.1 二分返回值与根误差保证
受审A1实验公共代码bisect在收缩后的新区间判停，却返回旧hist；A1实验5.1-A实际把hist[-1][3]作为近似根。测试用高精度Decimal核查严格根误差，避免用旧输出作为真值。

In [2]:
A1_COMMON='import numpy as np\n\nnp.set_printoptions(precision=6, suppress=True)\n\n\ndef bisect(f, a, b, tol=1e-12, maxit=200):\n    """二分法：要求 f(a)*f(b) < 0。返回 (迭代历史, 是否达标)。"""\n    fa, fb = f(a), f(b)\n    if fa * fb >= 0:\n        raise ValueError("区间端点必须异号")\n    hist = []\n    for k in range(maxit):\n        if (b - a) / 2 <= tol:\n            return hist, True\n        m = 0.5 * (a + b)\n        fm = f(m)\n        hist.append((k, a, b, m, fm))\n        if fm == 0:\n            return hist, True\n        if fa * fm < 0:\n            b, fb = m, fm\n        else:\n            a, fa = m, fm\n    return hist, False\n\n\ndef fixed_point(g, x0, tol=1e-12, maxit=200):\n    """不动点迭代 x_{k+1}=g(x_k)。"""\n    x, hist = x0, [x0]\n    for _ in range(maxit):\n        x = g(x)\n        hist.append(x)\n        if abs(hist[-1] - hist[-2]) <= tol:\n            return np.array(hist), True\n    return np.array(hist), False\n\n\ndef newton1d(f, df, x0, tol=1e-12, maxit=50):\n    """标量 Newton 法。"""\n    x, hist = x0, [x0]\n    for _ in range(maxit):\n        if df(x) == 0:\n            return np.array(hist), False\n        x = x - f(x) / df(x)\n        hist.append(x)\n        if abs(f(x)) <= tol:\n            return np.array(hist), True\n    return np.array(hist), False\n\n\ndef armijo(f, grad, x0, alpha0=1.0, c1=1e-4, rho=0.5, tol=1e-8, maxit=10000):\n    """负梯度方向 + Armijo 回溯。返回 (轨迹, 步长序列, 函数评估次数, 是否达标)。"""\n    x = np.asarray(x0, dtype=float).copy()\n    hist, alphas, fev = [x.copy()], [], 0\n    for _ in range(maxit):\n        fx, g = f(x), grad(x)\n        if np.linalg.norm(g) <= tol:\n            return np.array(hist), alphas, fev, True\n        d, a = -g, alpha0\n        while f(x + a * d) > fx + c1 * a * (g @ d):\n            a *= rho\n            fev += 1\n            if a < 1e-16:\n                return np.array(hist), alphas, fev, False\n        fev += 1\n        alphas.append(a)\n        x = x + a * d\n        hist.append(x.copy())\n    return np.array(hist), alphas, fev, False\n'

In [3]:
from decimal import Decimal,localcontext
ns={'np':np};tree=ast.parse(A1_COMMON);tree.body=[n for n in tree.body if isinstance(n,ast.FunctionDef)];exec(compile(tree,'A1公共原定义','exec'),ns)
hist,ok=ns['bisect'](lambda x:x*x-2,1.,2.,tol=1e-12)
value=hist[-1][3]
with localcontext() as ctx:
 ctx.prec=70;root=Decimal(2).sqrt();error=float(abs(Decimal.from_float(value)-root))
rec('C001_bisection',dict(ok=bool(ok),returned_last_midpoint=value,error=error,tolerance=1e-12,violates=error>1e-12,updates=len(hist)))
assert ok and error>1e-12
errors=[float(abs(Decimal.from_float(h[3])-root)) for h in hist]
bounds=[(h[2]-h[1])/2 for h in hist]
assert all(e<=b+1e-16 for e,b in zip(errors,bounds))
plt.semilogy(errors,label='actual root error');plt.semilogy(bounds,label='half-width of recorded interval');plt.axhline(1e-12,color='k',ls=':');plt.legend();plt.xlabel('record index');plt.ylabel('error');plt.tight_layout();plt.savefig(OUT/'C1组-G8-二分误差.png');plt.close()
xs=[1.]
for k in range(7):xs.append(xs[-1]/2+1/xs[-1])
cycle=ns['newton1d'](lambda x:x**3-2*x+2,lambda x:3*x*x-2,0.,maxit=6)
rec('Newton_conditions',dict(sqrt2_errors=[abs(x-math.sqrt(2)) for x in xs],two_cycle=cycle[0].tolist(),cycle_success=bool(cycle[1]),multiplicity2_ratio=.5))

C001_bisection {
  "ok": true,
  "returned_last_midpoint": 1.4142135623715149,
  "error": 1.5801947803183132e-12,
  "tolerance": 1e-12,
  "violates": true,
  "updates": 39
}
Newton_conditions {
  "sqrt2_errors": [
    0.41421356237309515,
    0.08578643762690485,
    0.002453104293571373,
    2.123901414519125e-06,
    1.5947243525715749e-12,
    2.220446049250313e-16,
    2.220446049250313e-16,
    2.220446049250313e-16
  ],
  "two_cycle": [
    0.0,
    1.0,
    0.0,
    1.0,
    0.0,
    1.0,
    0.0
  ],
  "cycle_success": false,
  "multiplicity2_ratio": 0.5
}


## 5.2 B1稳定性状态标签与Wolfe证明
B1验证Notebook零基单元0用5步后的范数是否大于10来判发散。对A=diag(2,8)、α=0.3，高频模态因子1-8α=-1.4，已经是发散迭代；有限五步大小不足以证明收敛。另以ψ(α)=exp(-α)反驳“A1 Wolfe证明中下有界加初始下降必取得有限局部极小”的中间推理；不否定Wolfe存在定理本身。

In [4]:
A=np.diag([2.,8.]);x=np.ones(2);norms=[float(np.linalg.norm(x))]
for k in range(30):x=x-.3*A@x;norms.append(float(np.linalg.norm(x)))
label='发散' if norms[5]>10 else '收敛中'
rec('C002_B1_label',dict(alpha=.3,eigen_factors=[.4,-1.4],spectral_radius=1.4,norm5=norms[5],B1_rule_label=label,norm30=norms[30],conclusion='除去不稳定特征方向初分量为零的特殊初值，一般发散；当前初值有该分量'))
assert label=='收敛中' and norms[30]>1e4
plt.semilogy(norms,'o-');plt.xlabel('k');plt.ylabel('norm(x_k), alpha=0.3');plt.tight_layout();plt.savefig(OUT/'C1组-G8-稳定性标签.png');plt.close()
c1,c2=1e-4,.9;alpha=-math.log(c2)
rec('C003_Wolfe_proof',dict(psi='exp(-alpha)',derivative='-exp(-alpha)<0 for every finite alpha',bounded_below=0,finite_stationary_point=False,wolfe_alpha=alpha,armijo=math.exp(-alpha)<=1-c1*alpha,strong_wolfe=abs(-math.exp(-alpha))<=c2,conclusion='否定错误中间推理；此例仍有Wolfe步长'))
rec('Armijo_not_curvature',dict(phi='-x^2/2',x=1,p=1,alpha=1,s_y=-1,armijo=True,conclusion='仅Armijo不保证BFGS曲率条件；A2正文已明确说明，非A2错误'))

C002_B1_label {
  "alpha": 0.3,
  "eigen_factors": [
    0.4,
    -1.4
  ],
  "spectral_radius": 1.4,
  "norm5": 5.378249748310318,
  "B1_rule_label": "收敛中",
  "norm30": 24201.432355484558,
  "conclusion": "除去不稳定特征方向初分量为零的特殊初值，一般发散；当前初值有该分量"
}
C003_Wolfe_proof {
  "psi": "exp(-alpha)",
  "derivative": "-exp(-alpha)<0 for every finite alpha",
  "bounded_below": 0,
  "finite_stationary_point": false,
  "wolfe_alpha": 0.10536051565782628,
  "armijo": true,
  "strong_wolfe": true,
  "conclusion": "否定错误中间推理；此例仍有Wolfe步长"
}
Armijo_not_curvature {
  "phi": "-x^2/2",
  "x": 1,
  "p": 1,
  "alpha": 1,
  "s_y": -1,
  "armijo": true,
  "conclusion": "仅Armijo不保证BFGS曲率条件；A2正文已明确说明，非A2错误"
}


## 5.3 Newton方向与BFGS单次更新
H=diag(-1,2)、g=(1,0)时Newton方向不是下降方向。另用B=I,s=(1,0),y=(2,1)核查BFGS割线与正定性；这不是一般收敛证明。

In [5]:
H=np.diag([-1.,2.]);g=np.array([1.,0.]);p=np.linalg.solve(H,-g)
B=np.eye(2);s=np.array([1.,0.]);y=np.array([2.,1.]);bs=B@s
Bnew=B-np.outer(bs,bs)/(s@bs)+np.outer(y,y)/(s@y)
rec('BFGS',dict(newton_gTp=float(g@p),Bnew=Bnew.tolist(),secant_error=float(np.linalg.norm(Bnew@s-y)),eigenvalues=np.linalg.eigvalsh(Bnew).tolist()))
assert g@p>0 and np.linalg.norm(Bnew@s-y)<1e-12 and np.linalg.eigvalsh(Bnew).min()>0

BFGS {
  "newton_gTp": 1.0,
  "Bnew": [
    [
      2.0,
      1.0
    ],
    [
      1.0,
      1.5
    ]
  ],
  "secant_error": 0.0,
  "eigenvalues": [
    0.719223593595585,
    2.7807764064044154
  ]
}


## 5.4 独立GN/LM一步、存在性及非零残差反例
A1指数例使用t=[0,1,2],y=[1,2.5,6],初值(a,b)=(1,0.5)。用SVD最小二乘和增广最小二乘独立核查B1/B2给出的正规方程步；残差r=y-a exp(bt)，更新θ+d。

In [6]:
t=np.array([0.,1.,2.]);obs=np.array([1.,2.5,6.]);a,b=1.,.5;E=np.exp(b*t);r=obs-a*E;J=np.column_stack([-E,-a*t*E])
dgn=np.linalg.lstsq(J,-r,rcond=None)[0]
dgn_normal=np.linalg.solve(J.T@J,-J.T@r)
damp=.1
dlm=np.linalg.lstsq(np.vstack([J,np.sqrt(damp)*np.eye(2)]),np.r_[-r,np.zeros(2)],rcond=None)[0]
assert max(abs(dgn-dgn_normal))<1e-12
def objective(theta):return .5*np.linalg.norm(obs-theta[0]*np.exp(theta[1]*t))**2
eps=1e-6;fd=np.array([(objective(np.array([a,b])+eps*np.eye(2)[i])-objective(np.array([a,b])-eps*np.eye(2)[i]))/(2*eps) for i in range(2)])
rec('C004_GN_LM',dict(residual=r.tolist(),Jacobian=J.tolist(),GN_step=dgn.tolist(),LM_step=dlm.tolist(),gradient_fd_error=float(np.linalg.norm(fd-J.T@r)),reference='独立SVD/增广最小二乘；与正规方程容差1e-12',B1_B2_steps_confirmed=True))
# Linear model LM lambda=1: exact linearization gives actual/predicted reduction ratio 1.
M=np.column_stack([np.ones(3),t]);yy=np.array([1.,2.,4.]);d=np.linalg.solve(M.T@M+np.eye(2),M.T@yy)
F0=.5*yy@yy;F1=.5*np.linalg.norm(yy-M@d)**2;pred=F0-.5*np.linalg.norm(yy-M@d)**2
rec('LM_linear_ratio',dict(step=d.tolist(),F1=float(F1),rho=float((F0-F1)/pred)))
rec('C005_existence',dict(residual='r(x)=exp(x)',objective='F(x)=exp(2x)/2',lower_bound=0,minimum_attained=False,samples=[dict(x=x,F=.5*math.exp(2*x)) for x in [0,-5,-10]],conclusion='有下界不保证有限极小点存在'))
rec('nonzero_residual_counterexample',dict(residual='r(x)=(x,1)',J=[[1],[0]],S=0,GN_update='x_next=0',residual_at_solution=[0,1],conclusion='非零残差不推出S非零，也不必然排除更快收敛；教学构造'))
rec('rank_deficiency',dict(residual='r(a,b)=a+b-1',J=[[1,1]],J_rank=1,solutions='a+b=1整条直线',conclusion='小残差不保证参数唯一；LM正则化当前步不补充观测信息'))
print('复核基准执行完成。')

C004_GN_LM {
  "residual": [
    0.0,
    0.8512787292998718,
    3.281718171540955
  ],
  "Jacobian": [
    [
      -1.0,
      -0.0
    ],
    [
      -1.6487212707001282,
      -1.6487212707001282
    ],
    [
      -2.718281828459045,
      -5.43656365691809
    ]
  ],
  "GN_step": [
    -0.06698626476266409,
    0.6325986533383502
  ],
  "LM_step": [
    -0.04364425053690451,
    0.6180297533424824
  ],
  "gradient_fd_error": 5.51463665419425e-10,
  "reference": "独立SVD/增广最小二乘；与正规方程容差1e-12",
  "B1_B2_steps_confirmed": true
}
LM_linear_ratio {
  "step": [
    0.8,
    1.2666666666666666
  ],
  "F1": 0.24444444444444463,
  "rho": 1.0
}
C005_existence {
  "residual": "r(x)=exp(x)",
  "objective": "F(x)=exp(2x)/2",
  "lower_bound": 0,
  "minimum_attained": false,
  "samples": [
    {
      "x": 0,
      "F": 0.5
    },
    {
      "x": -5,
      "F": 2.2699964881242427e-05
    },
    {
      "x": -10,
      "F": 1.030576811219279e-09
    }
  ],
  "conclusion": "有下界不保证有限极小点存在"
}
nonzero

## 本轮运行记录

2026年11月21日，使用 Python 3.12.14、NumPy 1.26.4、SciPy 1.14.1、Matplotlib 3.9.2，在同一独立命名空间按单元顺序执行全部 6 个代码单元，全部断言通过。输出由本轮执行生成。运行范围为本文件中的复核基准及内嵌的 A1 公共函数；没有重新执行完整 A/B 实验。